In [51]:
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
from LINZ.Geodetic import GDB
from LINZ.Geodetic.Ellipsoid import GRS80

import datetime
import pandas as pd
import csv
import numpy as np
import requests
import json
from astropy.time import Time
import re

In [244]:
#All normal user inputs should be in this box
start=datetime.datetime(2025,3,20) #format = YYYY,M(M),D(D)
end=datetime.datetime(2025,3,26) #format = YYYY,M(M),D(D)

solutiontype = 'd20f_52_code_A'   #This is the solution used in the daily processing
itrf_input_occ_code = "LINZ:ITRF2020_XYZ"  #this is the code to tell the Online Coordinate Converter the input coordsys of the conversions

lol_adj_desc = "2025.02.01 Private CORS Update (DefMod v20180701 ITRF2020@2023-07-01)" # Make sure it has enough info so future users can find the misc job folder for this calculation
nzgd2000_order = "0" # The order for the NZGD2000 output coordinates
nzvd2016_order = "1V" # The older for the NZVD2016 output coordinates

# Two options to get list of marks for this calculation, # out the one you don't use:
# 1. Add geodetic codes of marks for the calculation to the codes list
codes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG','VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']


In [245]:
### Print out a list of important parameters based on the input.

crd_epoch = end + (start - end) / 2  #this is the epoch all the resulting coordinates will be in.  I like it to be 1 Jan, but you could pick any date.
print (f'crd_epoch: {crd_epoch} --- nominal coordinate epoch of the output coordinates\n')
#input_date =  datetime.datetime(2007, 4, 14, 11, 42, 50)

astropy_time_object = Time(crd_epoch,format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear
print (f'crd_epoch_decimal_year : {crd_epoch_decimal_year} --- as above but in decimal year format used in Online Coordinate Converter conversions\n')

ndays = (end-start).days
print (f'ndays: {ndays} --- the number of days from start date to end date\n')
print (f'ndays/2: {int(ndays/2)} --- every mark must have at least this many days of data\n') 
print (f'ndays/8:  {int(ndays/8)} --- every mark must have at least this many days of data before and after the crd_epoch\n')
#print("crd_epoch : " + crd_epoch.strftime("%d/%m/%Y %H:%M") + ' --- this is the nominal epoch of the calculated coordinates and the mid-date\n')

count = len(codes)
print (f'number of cors: {count} \n')
#for c in codes:
#    print (c, type(c))

crd_epoch: 2025-03-23 00:00:00 --- nominal coordinate epoch of the output coordinates

crd_epoch_decimal_year : 2025.2219178082191 --- as above but in decimal year format used in Online Coordinate Converter conversions

ndays: 6 --- the number of days from start date to end date

ndays/2: 3 --- every mark must have at least this many days of data

ndays/8:  0 --- every mark must have at least this many days of data before and after the crd_epoch

number of cors: 36 



In [246]:
### Set up dataframe to receive regression calculation error messages

df_regres_error = pd.DataFrame (codes, columns = ['siteid'])
df_regres_error['has_data'] = None
df_regres_error['calcd_crd'] = None
df_regres_error['comment'] = None
#df_regres_error

In [247]:
df_model_ITRF_crds = pd.DataFrame()

In [248]:
for code in codes:
    #print (code)


    try:
        print (code, solutiontype)
        ts=CoordApiTimeseries(code,solutiontype = solutiontype,after=start,before=end) #this creates a timeseries object
    except:
        print ("timeseries object failed to create")
    try:
        #print (ts)
        dates,xyz=ts.withoutOutliers().getObs(enu=False)                                  #this removes outliers
        #print (xyz)

        # This gives every day from the start day a consecutive number starting from 1
        # It counts how many days of data there are and how many days before and after the mid-epoch
        num_day = (dates-start).astype('timedelta64[D]').astype(int) 
        num_day_size = num_day.size
        days_after_epoch = num_day[num_day > ndays/2].size
        days_before_epoch = num_day[num_day < ndays/2].size

        # if the number of days exceeds the arbitarily chosen values, then the regression coordinates are calculated
        if num_day_size > ndays/2 and days_after_epoch > ndays/8 and days_before_epoch > ndays/8:
            #print ('passes count')
            # Calculate regression and coordinates
            # 'dayno' is basically the date of each coordinate.  
            dayno = num_day.reshape((num_day.size,1)) #changed from 1d to 2d array where the second d is 1
            n = np.size(dayno)
            dayno_mean = dayno.mean()
            xyz_mean = xyz.mean(axis=0)  # This is an array of size 3.
            #print (xyz_mean)
            Sxy = np.sum(dayno*xyz, axis=0)- n*dayno_mean*xyz_mean
            Sxx = np.sum(dayno*dayno, axis=0)-n*dayno_mean*dayno_mean

            # b1 = slope
            b1 = Sxy/Sxx

            # b0 = y-intercept
            b0 = xyz_mean-b1*dayno_mean

            y_pred = b1 * (ndays/2) + b0
            new_row = pd.DataFrame([[code,*y_pred]],columns=['code','x','y','z'])# *y_pred each of the values in this iterable thing
            #print (new_row)
            df_model_ITRF_crds = pd.concat([df_model_ITRF_crds, new_row], ignore_index=True)
            df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'y'
            df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'y'

        # if anything above fails it gets picked up in this 'else' or the 'except' below
        else:
            #print (f'not enough coordinates for station {code}')
            df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'n'
            df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'y'
            df_regres_error.loc[df_regres_error.siteid == code, 'comment'] = f'not enough coordinates for station {code}'

    except Exception as ex:
        #print (f"couldn't get coords for station {code} {ex}")
        df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'n'
        df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'n'
        df_regres_error.loc[df_regres_error.siteid == code, 'comment'] = f"couldn't get coords for station {code} {ex}"
        continue

AUCK d20f_52_code_A
BLUF d20f_52_code_A
CHTI d20f_52_code_A
CORM d20f_52_code_A
DNVK d20f_52_code_A
DUND d20f_52_code_A
GISB d20f_52_code_A
GLDB d20f_52_code_A
HAAS d20f_52_code_A
HAMT d20f_52_code_A
HAST d20f_52_code_A
HIKB d20f_52_code_A
HOKI d20f_52_code_A
KAIK d20f_52_code_A
KTIA d20f_52_code_A
LEXA d20f_52_code_A
LKTA d20f_52_code_A
MAHO d20f_52_code_A
MAVL d20f_52_code_A
METH d20f_52_code_A
MQZG d20f_52_code_A
MTJO d20f_52_code_A
NLSN d20f_52_code_A
NPLY d20f_52_code_A
PYGR d20f_52_code_A
TAUP d20f_52_code_A
TRNG d20f_52_code_A
VGMT d20f_52_code_A
WAIM d20f_52_code_A
WANG d20f_52_code_A
WARK d20f_52_code_A
WEST d20f_52_code_A
WGTN d20f_52_code_A
WHKT d20f_52_code_A
WHNG d20f_52_code_A
WRPA d20f_52_code_A


In [249]:
##Send regression error information to csv

df_regres_error=df_regres_error.sort_values(['calcd_crd','has_data'])
df_regres_error.to_csv('regres_error_info.csv')
df_regres_error

,siteid,has_data,calcd_crd,comment
7,GLDB,y,n,not enough coordinates for station GLDB
0,AUCK,y,y,None
1,BLUF,y,y,None
2,CHTI,y,y,None
3,CORM,y,y,None
4,DNVK,y,y,None
5,DUND,y,y,None
6,GISB,y,y,None
8,HAAS,y,y,None
9,HAMT,y,y,None


In [250]:
print (df_model_ITRF_crds)

    code             x             y             z
0   AUCK -5.105682e+06  4.615640e+05 -3.782181e+06
1   BLUF -4.300031e+06  8.911137e+05 -4.610274e+06
2   CHTI -4.607856e+06 -2.723751e+05 -4.386954e+06
3   CORM -5.095051e+06  3.786672e+05 -3.805557e+06
4   DNVK -4.860761e+06  3.256930e+05 -4.103646e+06
5   DUND -4.388121e+06  7.266722e+05 -4.556533e+06
6   GISB -4.985376e+06  1.840222e+05 -3.960830e+06
7   HAAS -4.502928e+06  8.927841e+05 -4.414671e+06
8   HAMT -5.027290e+06  4.301778e+05 -3.888561e+06
9   HAST -4.912021e+06  2.809406e+05 -4.045418e+06
10  HIKB -5.060144e+06  1.498851e+05 -3.867002e+06
11  HOKI -4.635698e+06  7.355233e+05 -4.304158e+06
12  KAIK -4.685481e+06  5.310547e+05 -4.280819e+06
13  KTIA -5.190164e+06  6.121736e+05 -3.644201e+06
14  LEXA -4.421518e+06  8.347960e+05 -4.505701e+06
15  LKTA -4.646208e+06  6.309726e+05 -4.310354e+06
16  MAHO -4.977266e+06  4.482295e+05 -3.950347e+06
17  MAVL -4.392931e+06  9.242781e+05 -4.516480e+06
18  METH -4.577297e+06  6.77933

In [251]:
##Send global CORS ITRF calculated coordinates to csv

df_model_ITRF_crds.to_csv('global_cors_itrf_calc_coords.csv', index=False)

In [252]:
##Get a list of all marks we have ITRF2014 modelled coordinates for ready to query the GDB

mark_list = df_model_ITRF_crds['code'].tolist()
#print (mark_list)

In [253]:
columns=["code","gdb_long","gdb_lat","gdb_ellhgt"]
gdbdata=[]
for m in mark_list:
    #print (m)
    try:
        mark=GDB.get(m)
        c=mark.coordinate
        if mark.mark_data.country != 'NEWZ':
            continue
        gdbdata.append([m,c.longitude,c.latitude,c.height])
    except Exception as ex:
        #gdbdata.append([m,0.0,0.0,0.0,False])
        print (f"couldn't get coords for station {m} {ex}")
        continue
        
# df has the same rows as df, including isvalid column identifying which we are interested in
df_gdb_nzgd2000=pd.DataFrame(gdbdata,columns=columns)
df_gdb_nzgd2000

,code,gdb_long,gdb_lat,gdb_ellhgt
0,AUCK,174.834385,-36.602845,132.6906
1,BLUF,168.292087,-46.585064,124.6495
2,CHTI,-176.617110,-43.735477,75.6776
3,CORM,175.749557,-36.865433,170.2451
4,DNVK,176.166658,-40.298858,457.6126
5,DUND,170.597170,-45.883666,386.9323
6,GISB,177.886035,-38.635337,87.1712
7,HAAS,168.785553,-44.073206,1053.5668
8,HAMT,175.109198,-37.806755,69.4115
9,HAST,176.726564,-39.617033,152.3211


In [254]:
occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to" 

input_crds = df_model_ITRF_crds[['x', 'y', 'z']].values.tolist() #This makes a list of x,y,z cooordinate lists for input so API is queried once.

crds_list=[]
print (crd_epoch_decimal_year, "is the epoch of the conversion")
coordlist = {
"crs": itrf_input_occ_code,
"coordinateOrder": ["geocentricX","geocentricY","geocentricZ"],
"coordinateEpoch": crd_epoch_decimal_year,
"coordinates": input_crds,
}
params = {"crs": "LINZ:NZGD2000" }

response = requests.post(occ_api, params=params, json=coordlist)
if response.status_code == 200:
    converted = response.json()
    crds_list = converted['coordinateList']['coordinates']

    #print(json.dumps(nzgdcoords, indent=2))
else:
    print(f"Error: {response.text}")

2025.2219178082191 is the epoch of the conversion


In [255]:
zipped_list = []
for code,crd in zip(df_model_ITRF_crds.code,crds_list): 
    if crd is not None:
        zipped_list.append([code,crd[0],crd[1],crd[2]])
        #print (idx,code,crd)
        
df_regres_converted_to_nzgd2000 = pd.DataFrame(zipped_list,columns=("code","mlat","mlong","mellhgt"))     

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#    display(df_regres_converted_to_nzgd2000)
    
    
#df_regres_converted_to_nzgd2000  

In [256]:
df_combine = pd.merge(df_gdb_nzgd2000, df_regres_converted_to_nzgd2000, how='inner', left_on = 'code', right_on = 'code')
#df_combine

In [257]:
### Note: You can do these calcs more like (but thinking you want to see computation steps)
### Also this is quick enough there is no reason to use more approximate method below.
### Useful for understanding and how to quickly approximate, but not necessary in this case.

dedlon,dndlat=GRS80.metres_per_degree(df_combine.gdb_long,df_combine.gdb_lat)
df_combine['d_east']=(df_combine.mlong-df_combine.gdb_long)*dedlon
df_combine['d_north']=(df_combine.mlat-df_combine.gdb_lat)*dndlat
df_combine['d_horz']=np.sqrt(((df_combine.mlong-df_combine.gdb_long)*dedlon)**2 + ((df_combine.mlat-df_combine.gdb_lat)*dndlat)**2)
df_combine['d_ellhgt']=df_combine.mellhgt-df_combine.gdb_ellhgt
df_combine

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt
0,AUCK,174.834385,-36.602845,132.6906,-36.602845,174.834385,132.6901,-0.004742,0.001110,0.004870,-0.0005
1,BLUF,168.292087,-46.585064,124.6495,-46.585064,168.292086,124.6513,-0.005365,0.000333,0.005375,0.0018
2,CHTI,-176.617110,-43.735477,75.6776,-43.735477,-176.617110,75.6736,0.002739,0.000333,0.002759,-0.0040
3,CORM,175.749557,-36.865433,170.2451,-36.865433,175.749557,170.2453,-0.006063,0.004550,0.007581,0.0002
4,DNVK,176.166658,-40.298858,457.6126,-40.298858,176.166658,457.6072,0.002551,0.005774,0.006312,-0.0054
5,DUND,170.597170,-45.883666,386.9323,-45.883666,170.597170,386.9302,-0.003338,0.002112,0.003950,-0.0021
6,GISB,177.886035,-38.635337,87.1712,-38.635337,177.886035,87.1653,0.011232,-0.004884,0.012248,-0.0059
7,HAAS,168.785553,-44.073206,1053.5668,-44.073205,168.785553,1053.5692,-0.002804,0.001444,0.003154,0.0024
8,HAMT,175.109198,-37.806755,69.4115,-37.806755,175.109198,69.4145,-0.005020,0.001332,0.005193,0.0030
9,HAST,176.726564,-39.617033,152.3211,-39.617033,176.726564,152.3163,-0.011163,0.000000,0.011163,-0.0048


In [258]:
specified_value_test1 = 0.025
df_combine[abs(df_combine.d_horz) > specified_value_test1]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [259]:
specified_value_test2 = 0.025
df_combine[abs(df_combine.d_ellhgt) > specified_value_test2]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [260]:
specified_value_test3 = 0.05
df_combine[(abs(df_combine['d_ellhgt']) > specified_value_test3) & (abs(df_combine['d_horz']) > specified_value_test3)]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [261]:
specified_value_test4 = 0.05
df_combine[(abs(df_combine['d_ellhgt']) > specified_value_test4) | (abs(df_combine['d_horz']) > specified_value_test4)]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [262]:
df_combine.to_csv('df_combined_gdb_modelled.csv', index=False)
#df_model_ITRF_crds

In [263]:
df_model_ITRF_crds_list = df_model_ITRF_crds.values.tolist()
gdb_code_list = df_combine['code'].tolist()
#df_model_ITRF_crds_list

In [264]:
#This section creates the header information for the SNAP crd file.
doy = '%Y:%j'
ymd = '%Y%m%d'

#print (type(crd_epoch))
print ("crd_epoch:",crd_epoch)

crd_epoch_ymd = datetime.datetime.strftime(crd_epoch, ymd)
print ("crd_epoch_ymd:",crd_epoch_ymd)

start_doy = datetime.datetime.strftime(start, doy)
end_doy = datetime.datetime.strftime(end, doy)
today = datetime.date.today()
print ("today:",today)

#create name for snap_crds file = '20230702_d20f_52_code_A_xyz.crd'
snap_crds = '{}_{}_xyz.crd'.format (crd_epoch_ymd, solutiontype)
print ("output snap crd file name: ", snap_crds)

itrf = re.split('_|:', itrf_input_occ_code)[1]
crd_heading = 'PositioNZ coordinates calculated on {} by averaging daily solutions from {} to {} @nominal epoch {}\n'.format (today, start_doy, end_doy, crd_epoch_ymd)
crd_system =  '{}_XYZ@{}\n'.format(itrf,crd_epoch_ymd)
crd_options = 'options no_geoid\n'
print (crd_heading)
print (crd_system)
print (crd_options)

#This section writes the SNAP crd file that contains the mean coordinates of the period queried.
with open(snap_crds, 'w')as out2:
    out2.write(crd_heading)
    out2.write(crd_system)
    out2.write(crd_options)
    out2.write("\n")
    
    for code_xyz in df_model_ITRF_crds_list:
        list_code = code_xyz[0]
        list_x = "{:.5f}".format(code_xyz[1])
        list_y = "{:.5f}".format(code_xyz[2])
        list_z = "{:.5f}".format(code_xyz[3])
        if list_code in gdb_code_list:
            out2.write('{}  {}  {}  {}\n'.format(list_code,list_x,list_y,list_z))

crd_epoch: 2025-03-23 00:00:00
crd_epoch_ymd: 20250323
today: 2025-05-08
output snap crd file name:  20250323_d20f_52_code_A_xyz.crd
PositioNZ coordinates calculated on 2025-05-08 by averaging daily solutions from 2025:079 to 2025:085 @nominal epoch 20250323

ITRF2020_XYZ@20250323

options no_geoid



In [265]:
#df_regres_converted_to_nzgd2000

occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to" 

input_crds = df_model_ITRF_crds[['x', 'y', 'z']].values.tolist() #This makes a list of x,y,z cooordinate lists for input so API is queried once.

crds_list_v=[]

coordlist = {
"crs": itrf_input_occ_code,
"coordinateOrder": ["geocentricX","geocentricY","geocentricZ"],
"coordinateEpoch": crd_epoch_decimal_year,
"coordinates": input_crds,
}
params = {"crs": "LINZ:NZGD2000/NZVD2016"}

response = requests.post(occ_api, params=params, json=coordlist)
if response.status_code == 200:
    converted = response.json()
    crds_list_v = converted['coordinateList']['coordinates']

    #print(json.dumps(nzgdcoords, indent=2))
else:
    print(f"Error: {response.text}")
zipped_list_v = []
for code,crd in zip(df_model_ITRF_crds.code,crds_list_v): 
    #print (code,crd)
    if crd is not None:
        #zipped_list_v.append([code,crd[0],crd[1],crd[2]])
        zipped_list_v.append([code,crd[2]])
        #print (idx,code,crd)
        
df_regres_converted_to_nzvd2016 = pd.DataFrame(zipped_list_v,columns=("code","hgt"))     

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#    display(df_regres_converted_to_nzgd2000)
    
#df_regres_converted_to_nzvd2016['markid']=""
df_regres_converted_to_nzvd2016.insert(1, "markid", np.nan)
df_regres_converted_to_nzvd2016.insert(2, "crdsys", "NZVD2016")
df_regres_converted_to_nzvd2016.insert(3, "lon", np.nan)
df_regres_converted_to_nzvd2016.insert(4, "lat", np.nan)
df_regres_converted_to_nzvd2016.insert(6, "order", nzvd2016_order)
df_regres_converted_to_nzvd2016.insert(7, "description", lol_adj_desc)

#df_regres_converted_to_nzvd2016     

In [266]:
df_horz_crds_lol=df_regres_converted_to_nzgd2000[['code','mlat','mlong','mellhgt']].copy()

df_horz_crds_lol.insert(1, "markid", np.nan)
df_horz_crds_lol.insert(2, "crdsys", "NZGD2000")
df_horz_crds_lol.insert(6, "order", nzgd2000_order)
df_horz_crds_lol.insert(7, "description", lol_adj_desc)

df_horz_crds_lol = df_horz_crds_lol.rename(columns={'mlat': 'lat', 'mlong': 'lon', 'mellhgt': 'hgt'})
df_horz_crds_lol = df_horz_crds_lol[['code','markid','crdsys','lon','lat','hgt','order','description']]

df_output_to_load_to_lol = pd.concat([df_horz_crds_lol,df_regres_converted_to_nzvd2016])  
df_output_to_load_to_lol.to_csv('df_output_to_load_to_lol.csv', index=False)
df_output_to_load_to_lol

,code,markid,crdsys,lon,lat,hgt,order,description
0,AUCK,NaN,NZGD2000,174.834385,-36.602845,132.6901,0,2025.02.01 Private CORS Update (DefMod v201807...
1,BLUF,NaN,NZGD2000,168.292086,-46.585064,124.6513,0,2025.02.01 Private CORS Update (DefMod v201807...
2,CHTI,NaN,NZGD2000,-176.617110,-43.735477,75.6736,0,2025.02.01 Private CORS Update (DefMod v201807...
3,CORM,NaN,NZGD2000,175.749557,-36.865433,170.2453,0,2025.02.01 Private CORS Update (DefMod v201807...
4,DNVK,NaN,NZGD2000,176.166658,-40.298858,457.6072,0,2025.02.01 Private CORS Update (DefMod v201807...
...,...,...,...,...,...,...,...,...
30,WEST,NaN,NZVD2016,NaN,NaN,648.3457,1V,2025.02.01 Private CORS Update (DefMod v201807...
31,WGTN,NaN,NZVD2016,NaN,NaN,12.9757,1V,2025.02.01 Private CORS Update (DefMod v201807...
32,WHKT,NaN,NZVD2016,NaN,NaN,204.4911,1V,2025.02.01 Private CORS Update (DefMod v201807...
33,WHNG,NaN,NZVD2016,NaN,NaN,134.9576,1V,2025.02.01 Private CORS Update (DefMod v201807...


In [267]:
##Print marks in original codes list that didn't get coordinates in the output

code_list = df_output_to_load_to_lol['code'].tolist()
print (list(set(codes).difference(code_list)))

['GLDB']
